In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

In [3]:
# Impact Regions
_polygons = (
    gpd.read_parquet(os.path.join(DATA_DIR, IMPACT_REGION_POLYGONS))
    .rename(columns={"hierid": "region"})
    .set_index("region")
    .set_crs(epsg=4326)  # Assuming the data is WGS-82.
)

# Socioeconomics
socioeconomics = xr.open_zarr(os.path.join(DATA_DIR, SOCIOECONOMICS_URI))
socioeconomics = socioeconomics.sel(year=2026)[
    ["pop", "gdppc", "iso3"]
]

In [25]:
# Define Forecast Months
fc_months = [8, 9, 10, 11, 12, 1]

In [4]:
def compute_area_weighted_mean(ds, lat_name="lat", lon_name="lon"):
    weights = np.cos(np.deg2rad(ds[lat_name]))
    weights.name = "weights"
    return ds.weighted(weights).mean((lat_name, lon_name))

In [5]:
POREALLAS_PARSED_ERA5_URI = "gs://poreallas-public-20260605/v20260731/parsed/era5.zarr"
ERA5_RECENT = "/home/emily_zuetell/projects/poreallas/data/parsed/26_era5_daily_running.zarr"
POREALLAS_PARSED_FORECAST_URI = "/home/emily_zuetell/projects/poreallas/data/parsed/08_ecmwf_parsed.zarr"

In [ ]:
era5 = xr.load_dataset(
        POREALLAS_PARSED_ERA5_URI,
        engine="zarr",
        backend_kwargs={"storage_options": {"token": "anon"}},
    )
era5_recent = xr.open_zarr(ERA5_RECENT).rename({'t2m': 'tas'})
era5_full = xr.merge([era5, era5_recent]) # Original data to 2025, add on recent 2026 ERA5 data

/tmp/ipykernel_5779/1573885855.py:7: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  era5_full = xr.merge([era5, era5_recent]) # Original data to 2025, add on recent 2026 ERA5 data
/tmp/ipykernel_5779/1573885855.py:7: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  era5_full = xr.merge([era5, era5_recent]) # Original data to 2025, add on recent 2026 

In [7]:
forecast = xr.open_zarr(
        POREALLAS_PARSED_FORECAST_URI)

In [28]:
# Analysis Period (1995-2027)
era5_analysis = era5.sel(time=slice("1995-01-01", "2014-12-31"))
era5_analysis = era5_analysis.sel(time=era5_analysis.time.dt.month.isin([fc_months])).compute()
forecast_analysis = forecast.sel(time=slice("2026-08-01", "2027-01-31")).compute()

In [36]:
era5_hot_days_per_year = ((era5_analysis['tas']-273.15) > 32).groupby("time.year").sum(dim="time")
forecast_hot_days_per_year = ((forecast_analysis['tas']-273.15) > 32).groupby("time.year").sum(dim="time")

In [37]:
fc_land = compute_area_weighted_mean(analysis_utils.land_only(forecast_hot_days_per_year))
era5_land = compute_area_weighted_mean(analysis_utils.land_only(era5_hot_days_per_year))

In [44]:
MEAN = (fc_land.sum(dim = 'year')/era5_land.mean(dim = 'year')).mean(dim = 'number')
SE = ((fc_land.sum(dim = 'year')/era5_land.mean(dim = 'year')).std(dim = 'number'))/np.sqrt(fc_land.sum(dim = 'year').shape[0])

In [46]:
print(f"Mean: {MEAN.values:.2f}, SE: {SE.values:.3f}")

Mean: 1.44, SE: 0.013
